In [3]:
import warnings

In [4]:
import scipy

In [5]:
scipy.optimize.brentq

<function scipy.optimize._zeros_py.brentq(f, a, b, args=(), xtol=2e-12, rtol=np.float64(8.881784197001252e-16), maxiter=100, full_output=False, disp=True)>

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

In [7]:
#Football Data
from statsbombpy import sb

#Optimization
from scipy.optimize import linear_sum_assignment

#Progress Bar
from tqdm import tqdm
warnings.filterwarnings("ignore", message="credentials were not supplied")

In [8]:
PITCH_X_MAX = 120
PITCH_Y_MAX = 80
MIN_STARTERS = 11
WATERMARK_NAME     = "Laila Amin"
WATERMARK_SUBTITLE = "Data Science & Business Analytics"
FULL_MATCH_THRESHOLD = 100

In [28]:
#Exploring available competitions
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

free_comps = sb.competitions()

comps = free_comps[free_comps["competition_gender"] == "male"].copy()

comps["season_start"]=comps["season_name"].str.split("/").str[0].astype(int)

comps = comps[comps["season_start"] >= 2014].copy()

In [29]:
#Count Matches Per Competition
match_counts = []

for comp_id, season_id in zip(comps["competition_id"], comps["season_id"]):
    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
        matches_count = len(matches)
    except:
        matches_count = 0
        
    match_counts.append({
        "competition_id": comp_id,
        "season_id": season_id,
        "matches_count": matches_count
    })

match_counts_df = pd.DataFrame(match_counts)


metadata_cols = ["competition_id", "season_id", "competition_name", "season_name", "country_name"]
match_counts_df = match_counts_df.merge(
    comps[metadata_cols],
    on=["competition_id", "season_id"],
    how="left"
)

FULL_MATCH_THRESHOLD = 100  # leagues ~300+, cups smaller

full_comps = match_counts_df[
    match_counts_df["matches_count"] >= FULL_MATCH_THRESHOLD
].copy()


In [30]:
full_comps = full_comps.sort_values(
    "matches_count",
    ascending=False
).reset_index(drop=True)

full_comps

,competition_id,season_id,matches_count,competition_name,season_name,country_name
0,11,27,380,La Liga,2015/2016,Spain
1,12,27,380,Serie A,2015/2016,Italy
2,2,27,380,Premier League,2015/2016,England
3,7,27,377,Ligue 1,2015/2016,France
4,1238,108,115,Indian Super league,2021/2022,India


In [37]:
#Top 5 leagues
comps = sb.competitions()

TOP5 = ["Premier League", "La Liga", "Serie A", "1. Bundesliga", "Ligue 1", "Champions League"]

top5_comps = comps[
    (comps["competition_name"].isin(TOP5)) &
    (comps["season_name"] == "2014/2015")
][["competition_id", "season_id", "competition_name", "season_name", "country_name"]].reset_index(drop=True)

top5_comps

,competition_id,season_id,competition_name,season_name,country_name
0,16,26,Champions League,2014/2015,Europe
1,11,26,La Liga,2014/2015,Spain


In [44]:
#Fetching the Data
pairs = [(11,26), (16,26)]

all_matches = []
for comp, season in pairs:
    m = sb.matches(competition_id=comp, season_id=season)
    all_matches.append(m)

matches_df = pd.concat(all_matches, ignore_index=True)

barca_matches = matches_df[
    (matches_df["home_team"] == "Barcelona") | (matches_df["away_team"] == "Barcelona")
].sort_values("match_date").reset_index(drop=True)

len(barca_matches)

39

In [48]:
matches_df.head()

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,competition_id,competition_country_name,competition_name,competition,season_id,season,home_team_id,home_team,home_team_gender,home_team_group,home_team_country_id,home_team_country_name,away_team_id,away_team,away_team_gender,away_team_group,away_team_country_id,away_team_country_name,competition_stage_id,competition_stage,stadium_id,stadium,stadium_country_id,stadium_country_name,referee_id,referee,referee_country_id,referee_country_name,home_managers,away_managers,home_manager_id,home_manager_name,home_manager_nickname,home_manager_dob,home_manager_country_id,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,267183,2015-03-22,21:00:00.000,2,1,available,scheduled,2022-08-14T18:11:16.045589,2021-06-13T16:17:31.694,28,11,Spain,La Liga,Spain - La Liga,26,2014/2015,217,Barcelona,male,None,214,Spain,220,Real Madrid,male,None,214,Spain,1,Regular Season,342,Spotify Camp Nou,214,Spain,180.0,Antonio Miguel Mateu Lahoz,214.0,Spain,Luis Enrique Martínez García,Carlo Ancelotti,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,414.0,Carlo Ancelotti,NaN,1959-06-10,112.0,Italy,1.1.0,2,2
1,265835,2014-11-22,20:00:00.000,5,1,available,scheduled,2020-07-29T05:00,2021-06-13T16:17:31.694,12,11,Spain,La Liga,Spain - La Liga,26,2014/2015,217,Barcelona,male,None,214,Spain,213,Sevilla,male,None,214,Spain,1,Regular Season,342,Spotify Camp Nou,214,Spain,NaN,NaN,NaN,NaN,Luis Enrique Martínez García,Unai Emery Etxegoien,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,21.0,Unai Emery Etxegoien,Unai Emery,1971-11-03,214.0,Spain,1.1.0,2,2
2,266838,2014-09-24,22:00:00.000,0,0,available,scheduled,2020-07-29T05:00,2021-06-13T16:17:31.694,5,11,Spain,La Liga,Spain - La Liga,26,2014/2015,223,Málaga,male,None,214,Spain,217,Barcelona,male,None,214,Spain,1,Regular Season,346,Estadio La Rosaleda,214,Spain,NaN,NaN,NaN,NaN,Javier Gracia Carlos,Luis Enrique Martínez García,188.0,Javier Gracia Carlos,Javi Gracia,1970-05-01,214.0,Spain,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,1.1.0,2,2
3,265963,2015-04-28,20:00:00.000,6,0,available,scheduled,2022-08-14T18:34:52.948014,2021-06-13T16:17:31.694,34,11,Spain,La Liga,Spain - La Liga,26,2014/2015,217,Barcelona,male,None,214,Spain,216,Getafe,male,None,214,Spain,1,Regular Season,342,Spotify Camp Nou,214,Spain,222.0,David Fernández Borbalan,214.0,Spain,Luis Enrique Martínez García,Pablo Franco Martín,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,6277.0,Pablo Franco Martín,Pablo Franco,1973-10-16,214.0,Spain,1.1.0,2,2
4,266117,2014-09-27,18:00:00.000,6,0,available,scheduled,2020-07-29T05:00,2021-06-13T16:17:31.694,6,11,Spain,La Liga,Spain - La Liga,26,2014/2015,217,Barcelona,male,None,214,Spain,1049,Granada,male,None,214,Spain,1,Regular Season,342,Spotify Camp Nou,214,Spain,NaN,NaN,NaN,NaN,Luis Enrique Martínez García,Joaquín de Jesús Caparrós Camino,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,497.0,Joaquín de Jesús Caparrós Camino,Joaquín Caparrós,1955-10-15,214.0,Spain,1.1.0,2,2


In [52]:
#El Classico

el_classico = matches_df[
    (
        (matches_df["home_team"] == "Barcelona") &
        (matches_df["away_team"] == "Real Madrid") 
    )
    |
    (
        (matches_df["home_team"] == "Real Madrid") &
        (matches_df["away_team"] == "Barcelona")
    )
]
el_classico

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,competition_id,competition_country_name,competition_name,competition,season_id,season,home_team_id,home_team,home_team_gender,home_team_group,home_team_country_id,home_team_country_name,away_team_id,away_team,away_team_gender,away_team_group,away_team_country_id,away_team_country_name,competition_stage_id,competition_stage,stadium_id,stadium,stadium_country_id,stadium_country_name,referee_id,referee,referee_country_id,referee_country_name,home_managers,away_managers,home_manager_id,home_manager_name,home_manager_nickname,home_manager_dob,home_manager_country_id,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,267183,2015-03-22,21:00:00.000,2,1,available,scheduled,2022-08-14T18:11:16.045589,2021-06-13T16:17:31.694,28,11,Spain,La Liga,Spain - La Liga,26,2014/2015,217,Barcelona,male,None,214,Spain,220,Real Madrid,male,None,214,Spain,1,Regular Season,342,Spotify Camp Nou,214,Spain,180.0,Antonio Miguel Mateu Lahoz,214.0,Spain,Luis Enrique Martínez García,Carlo Ancelotti,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,414.0,Carlo Ancelotti,NaN,1959-06-10,112.0,Italy,1.1.0,2,2
14,267085,2014-10-25,18:00:00.000,3,1,available,scheduled,2022-08-14T17:28:42.264843,2021-06-13T16:17:31.694,9,11,Spain,La Liga,Spain - La Liga,26,2014/2015,220,Real Madrid,male,None,214,Spain,217,Barcelona,male,None,214,Spain,1,Regular Season,353,Bernabéu,214,Spain,183.0,Jesús Gil Manzano,214.0,Spain,Carlo Ancelotti,Luis Enrique Martínez García,414.0,Carlo Ancelotti,NaN,1959-06-10,112.0,Italy,793.0,Luis Enrique Martínez García,Luis Enrique,1970-05-08,214.0,Spain,1.1.0,2,2


In [53]:
all_passes = []
for match_id in tqdm(matches_df["match_id"], desc="Extracting passes"):
    events = sb.events(match_id=match_id)
    passes = events[events["type"] == "Pass"].copy()
    passes["match_id"] = match_id
    all_passes.append(passes)
passes_df = pd.concat(all_passes, ignore_index=True)

Extracting passes: 100%|██████████| 39/39 [00:38<00:00,  1.00it/s]


,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_offensive,foul_committed_type,foul_won_advantage,foul_won_defensive,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_technique,goalkeeper_type,id,index,interception_outcome,location,match_id,minute,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_deflected,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_outcome,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure,block_offensive,miscontrol_aerial_won,pass_straight,shot_open_goal,clearance_other,goalkeeper_shot_saved_to_post,pass_no_touch,shot_saved_to_post,foul_committed_penalty,foul_won_penalty,shot_one_on_one,goalkeeper_lost_in_play,pass_miscommunication,goalkeeper_success_in_play,injury_stoppage_in_chain,shot_redirect,goalkeeper_shot_saved_off_target,shot_saved_off_target,block_save_block,shot_follows_dribble,half_start_late_video_start,goalkeeper_punched_out,pass_backheel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.580000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92be8f5c-4bc3-4821-a140-1cafb7d99e65,5,NaN,"[61.0, 40.1]",267183,0,NaN,NaN,NaN,-2.371119,NaN,Right Foot,NaN,NaN,NaN,"[57.6, 36.8]",NaN,Ground Pass,NaN,4.738143,NaN,NaN,Neymar da Silva Santos Junior,4320.0,NaN,NaN,NaN,NaN,Kick Off,1,From Kick Off,Luis Alberto Suárez Díaz,5246.0,Center Forward,2,Barcelona,217,[863e6584-7b68-4c40-9e64-55c2caf8f8e3],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Barcelona,217,00:00:00.100,Pass,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.404755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76e45926-55a8-43b5-914e-27ca2b71d451,7,NaN,"[55.3, 37.5]",267183,0,NaN,NaN,NaN,2.827758,NaN,Right Foot,NaN,NaN,NaN,"[43.9, 41.2]",NaN,Ground Pass,NaN,11.985408,NaN,NaN,Javier Alejandro Mascherano,5506.0,NaN,NaN,NaN,NaN,NaN,1,From Kick Off,Neymar da Silva Santos Junior,4320.0,Left Wing,2,Barcelona,217,[34b4210f-53b3-4c9f-b05b-3fa56dc3475e],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Barcelona,217,00:00:00.680,Pass,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.430733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74aa8527-5156-4e36-b9ee-8c20836e7402,10,NaN,"[43.9, 41.2]",267183,0,NaN,NaN,NaN,-1.570796,NaN,Right Foot,NaN,NaN,NaN,"[43.9, 15.2]",NaN,Ground Pass,NaN,26.000000,NaN,NaN,Jordi Alba Ramos,5211.0,NaN,NaN,NaN,NaN,NaN,1,From Kick Off,Javier Alejandro Mascherano,5506.0,Center Defensive Midfield,2,Barcelona,217,[7b11983a-242a-4250-9d8f-18831ee2c4f5],2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Barcelona,217,00:00:02.289,Pass,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.632324,NaN,NaN,

In [56]:
matches_meta = matches_df[[
    "match_id", "match_date", "season", "competition",
    "home_team", "away_team", "home_score", "away_score"
]].copy()

passes_df = passes_df.merge(matches_meta, on="match_id", how="left")

passes_df.to_csv(
    "C:/Users/laila/Documents//passes_barcelona_201415_statsbomb_all.csv", index=False)

In [58]:
def clean_passes(passes_df: pd.DataFrame) -> pd.DataFrame:
    """Extract coordinates, remove invalid passes, and clip to pitch bounds."""
    passes = passes.df.copy()

    def _coords(col, idx):
        return col.apply(lambda v: v[idx] if isinstance(v, (list, tuple)) else None)

    passes["start_x"] = _coords(passes["location"], 0)
    passes["start_y"] = _coords(passes["location"], 1)
    passes["end_x"]   = _coords(passes["pass_end_location"], 0)
    passes["end_y"]   = _coords(passes["pass_end_location"], 1)

    passes = passes[passes["pass_outcome"] != "Injury Clearance"].copy()

    for col, max_val in [("start_x", PITCH_X_MAX), ("end_x", PITCH_X_MAX),
                         ("start_y", PITCH_Y_MAX), ("end_y", PITCH_Y_MAX)]:
        passes[col] = passes[col].clip(lower=0, upper=max_val)

    return passes

def load_passes(path: str) -> pd.DataFrame:
    """Load passes from a saved CSV and return
    a cleaned DataFrame ready for plotting."""

    import ast

    passes_df = pd.read_csv(
        path,
        low_memory=False,
        converters={
            "location":          lambda x: ast.literal_eval(x) if pd.notna(x) else None,
            "pass_end_location": lambda x: ast.literal_eval(x) if pd.notna(x) else None,
        }
    )

    return clean_passes(passes_df)

In [61]:
#Helpers
def _norm(x) -> str | None:
    """Lowercase and strip a player name for consistent matching."""
    return str(x).strip().lower() if pd.notna(x) else None

def format_nickname(name: str) -> str:
    """Capitalize each word of a name, preserving common particles."""
    if pd.isna(name):
        return name
    particles = {"de", "da", "del", "van", "von"}
    words = str(name).split()
    if all(w[0].isupper() or w.lower() in particles for w in words):
        return name
    return " ".join(w.lower() if w.lower() in particles else w.capitalize()
                    for w in words)

In [65]:
#Obtain staring XI
def get_starting_ix(match_id: int,
                    team: str) -> pd.DataFrame | None:
    try:
        lineup = sb.lineups(match_id=match_id)[team]
    except Exception:
        return None

    df = lineup.explode("positions")

    pos = pd.json_normalize(df["positions"])

    df = pd.concat(
        [df.drop(columns="positions").reset_index(drop=True), pos],
        axis=1
    )

    df["player"] = df["player_name"].apply(_norm)
    df["nickname"] = df["player_nickname"]
    df["from"] = df["from"].fillna("0:00")

    df = df.sort_values("from")

    starters = df.head(11)[["player", "nickname"]]

    return starters if len(starters) >= MIN_STARTERS else None

In [68]:
#Average Positions
def build_match_network(match_id: int, team: str, passes:pd.DataFrame):
    """
    Build average node positions for one team in one match.
    Returns (node_pos_df, pass_df, xi_df) or None if data is insufficient.
    """

    df = passes[
        (passes["match_id"] == match_id) & (passes["team"] == team)
    ].copy()

    if df.empty:
        return None
    df["p"] = df["player"].apply(_norm)
    df["f"] = df["pass_recipient"].apply(_norm)

    xi = get_starting_xi(match_id, team)

    if xi is None:
        return None

    starters = xi["player"].tolist()

    locs = pd.concat([
        df[df["p"].isin(starters)][["p", "start_x", "start_y"]].rename(
            columns={"p": "player", "start_x":"x", "start_y": "y"}
        ),
        df[df["r"].isin(starters)][["r", "end_x", "end_y"]].rename(
            columns={"r": "player", "end_x": "x", "end_y": "y"}
        ),

    ])

    avg = locs.groupby("player").agg(
        x=("x", "mean"),
        y=("y", "mean")
    ).reset_index()

    return (avg, df, xi) if len(avg) >= MIN_STARTERS else None